In [ ]:
import os
import io
import json
import pickle
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except:
    !pip install seaborn
    import seaborn as sns
try:
    import xgboost as xgb
except:
    !pip install xgboost
    import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, log_loss, classification_report,
    confusion_matrix
)

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_output = './output'
os.makedirs(str_output, exist_ok=True)

s3 = boto3.client('s3')

#### Functions

In [ ]:
def load_pickle_from_s3(bucket, key):
    """Download and deserialize a pickle object from S3."""
    response = s3.get_object(Bucket=bucket, Key=key)
    return pickle.loads(response['Body'].read())

#### Import Predictions and Model from S3

In [ ]:
# load test predictions from S3
df_preds = pd.read_csv(f's3://{str_bucket}/04_model/test_predictions.csv')

# load label encoder from S3
le = load_pickle_from_s3(str_bucket, '04_model/label_encoder.pkl')

# load model from S3
str_model_local = '/tmp/xgb_model.json'
s3.download_file(str_bucket, f'{str_task}/xgb_model.json', str_model_local)
model = xgb.Booster()
model.load_model(str_model_local)

# load params (local small file from 04_model)
with open('../04_model/output/params.json', 'r') as f:
    dict_params = json.load(f)

arr_classes = le.classes_
y_actual = df_preds['actual']
y_pred = df_preds['predicted']
arr_proba = df_preds[[f'prob_{c}' for c in arr_classes]].values

print(f'Test set size: {len(df_preds):,}')
print(f'Classes: {arr_classes}')

#### Overall Metrics

In [ ]:
y_actual_enc = le.transform(y_actual)

flt_accuracy = accuracy_score(y_actual, y_pred)
flt_logloss = log_loss(y_actual_enc, arr_proba)

print(f'Test Accuracy: {flt_accuracy:.4f}')
print(f'Test Log-Loss: {flt_logloss:.4f}')
print(f'Best iteration: {dict_params.get("best_iteration", "N/A")}')

#### Baselines Comparison

In [ ]:
# load train and test raw data from S3 to compute baselines
y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv').squeeze()
df_train_raw = pd.read_csv(f's3://{str_bucket}/02_data_split/train.csv')
df_test_raw = pd.read_csv(f's3://{str_bucket}/02_data_split/test.csv')

# baseline 1: always predict most common pitch type
str_most_common = y_train.mode()[0]
flt_baseline1 = accuracy_score(y_actual, [str_most_common] * len(y_actual))

# baseline 2: predict each pitcher's most common pitch type (from train)
sr_pitcher_mode = df_train_raw.groupby('pitcher_id')['pitch_type'].agg(lambda x: x.mode()[0])
arr_baseline2 = df_test_raw['pitcher_id'].map(sr_pitcher_mode).fillna(str_most_common)
flt_baseline2 = accuracy_score(y_actual, arr_baseline2)

# baseline 3: pitcher mode per count (from train)
df_train_raw['count_str'] = df_train_raw['balls'].astype(str) + '-' + df_train_raw['strikes'].astype(str)
df_test_raw['count_str'] = df_test_raw['balls'].astype(str) + '-' + df_test_raw['strikes'].astype(str)
sr_pitcher_count_mode = df_train_raw.groupby(['pitcher_id', 'count_str'])['pitch_type'].agg(lambda x: x.mode()[0])
arr_baseline3 = df_test_raw.set_index(['pitcher_id', 'count_str']).index.map(
    sr_pitcher_count_mode
)
arr_baseline3 = pd.Series(arr_baseline3).fillna(
    df_test_raw['pitcher_id'].map(sr_pitcher_mode).fillna(str_most_common)
).values
flt_baseline3 = accuracy_score(y_actual, arr_baseline3)

print(f'Baseline 1 - Always "{str_most_common}": {flt_baseline1:.4f}')
print(f'Baseline 2 - Pitcher mode:              {flt_baseline2:.4f}')
print(f'Baseline 3 - Pitcher mode per count:     {flt_baseline3:.4f}')
print(f'XGBoost model:                           {flt_accuracy:.4f}')
print(f'\nLift over best baseline: {(flt_accuracy - flt_baseline3)*100:.1f} percentage points')

#### Classification Report

In [ ]:
print(classification_report(y_actual, y_pred, target_names=arr_classes))

#### Confusion Matrix

In [ ]:
arr_cm = confusion_matrix(y_actual, y_pred, labels=arr_classes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# raw counts
sns.heatmap(arr_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=arr_classes, yticklabels=arr_classes, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# normalized by row (recall)
arr_cm_norm = arr_cm.astype(float) / arr_cm.sum(axis=1, keepdims=True)
sns.heatmap(arr_cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=arr_classes, yticklabels=arr_classes, ax=axes[1])
axes[1].set_title('Confusion Matrix (Row-Normalized / Recall)')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f'{str_output}/confusion_matrix.png', dpi=150)
plt.show()

#### Feature Importance

In [ ]:
# feature importance (gain)
dict_importance = model.get_score(importance_type='gain')
df_importance = pd.DataFrame({
    'feature': dict_importance.keys(),
    'gain': dict_importance.values()
}).sort_values('gain', ascending=False)

# top 30 features
fig, ax = plt.subplots(figsize=(10, 8))
df_top_feat = df_importance.head(30)
ax.barh(range(len(df_top_feat)), df_top_feat['gain'].values, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(df_top_feat)))
ax.set_yticklabels(df_top_feat['feature'].values)
ax.invert_yaxis()
ax.set_title('Top 30 Features by Gain')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.savefig(f'{str_output}/feature_importance.png', dpi=150)
plt.show()

#### Calibration Check

For each pitch type, compare predicted probability vs. actual frequency in bins.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, str_class in enumerate(arr_classes):
    ax = axes[i]
    arr_prob_class = arr_proba[:, i]
    arr_actual_class = (y_actual == str_class).astype(int).values
    
    # bin predictions into deciles
    df_cal = pd.DataFrame({'prob': arr_prob_class, 'actual': arr_actual_class})
    df_cal['bin'] = pd.qcut(df_cal['prob'], q=10, duplicates='drop')
    df_cal_grouped = df_cal.groupby('bin', observed=True).agg(
        mean_prob=('prob', 'mean'),
        mean_actual=('actual', 'mean'),
        count=('actual', 'size')
    )
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.scatter(df_cal_grouped['mean_prob'], df_cal_grouped['mean_actual'],
               s=df_cal_grouped['count'] / df_cal_grouped['count'].max() * 100,
               color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'{str_class}')
    ax.set_xlabel('Predicted Prob')
    ax.set_ylabel('Actual Freq')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle('Calibration Plots by Pitch Type', y=1.02)
plt.tight_layout()
plt.savefig(f'{str_output}/calibration_plots.png', dpi=150, bbox_inches='tight')
plt.show()

#### Prediction Confidence Distribution

In [ ]:
# max predicted probability (confidence) for each pitch
arr_max_prob = arr_proba.max(axis=1)
sr_correct = (y_actual.values == y_pred.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# distribution of confidence
axes[0].hist(arr_max_prob, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Max Predicted Probability')
axes[0].set_xlabel('Max Predicted Probability')
axes[0].set_ylabel('Count')

# accuracy by confidence bucket
df_conf = pd.DataFrame({'max_prob': arr_max_prob, 'correct': sr_correct})
df_conf['bin'] = pd.cut(df_conf['max_prob'], bins=10)
df_conf_agg = df_conf.groupby('bin', observed=True).agg(
    accuracy=('correct', 'mean'),
    count=('correct', 'size')
)
ax2 = axes[1]
x_labels = [f'{interval.left:.2f}-{interval.right:.2f}' for interval in df_conf_agg.index]
ax2.bar(range(len(df_conf_agg)), df_conf_agg['accuracy'], color='steelblue', edgecolor='black')
ax2.set_xticks(range(len(df_conf_agg)))
ax2.set_xticklabels(x_labels, rotation=45, ha='right')
ax2.set_title('Accuracy by Confidence Bucket')
ax2.set_xlabel('Max Predicted Probability')
ax2.set_ylabel('Accuracy')

plt.tight_layout()
plt.savefig(f'{str_output}/confidence_analysis.png', dpi=150)
plt.show()

#### Summary

In [ ]:
print('=' * 60)
print('MODEL EVALUATION SUMMARY')
print('=' * 60)
print(f'\nModel: XGBoost multi:softprob')
print(f'Best iteration: {dict_params.get("best_iteration", "N/A")}')
print(f'\n--- Test Set Metrics ---')
print(f'Accuracy:  {flt_accuracy:.4f}')
print(f'Log-Loss:  {flt_logloss:.4f}')
print(f'\n--- Baselines ---')
print(f'Always "{str_most_common}":       {flt_baseline1:.4f}')
print(f'Pitcher mode:             {flt_baseline2:.4f}')
print(f'Pitcher mode per count:   {flt_baseline3:.4f}')
print(f'\n--- Lift ---')
print(f'Over naive:               +{(flt_accuracy - flt_baseline1)*100:.1f} pp')
print(f'Over pitcher mode:        +{(flt_accuracy - flt_baseline2)*100:.1f} pp')
print(f'Over pitcher mode/count:  +{(flt_accuracy - flt_baseline3)*100:.1f} pp')
print('=' * 60)